In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("xgboost").setLevel(logging.ERROR)

print("Libraries Loaded")

In [ ]:
# PIPELINE CONFIG
# Set FORCE_REBUILD = True to reprocess raw files (slow, ~5 min)
# Set FORCE_REBUILD = False to load from cached CSVs (fast, <10 sec)

FORCE_REBUILD = True  # set to True whenever you change Cells 3 or 4

# ── Backtest window ──────────────────────────────────────────────────────────
# How many recent months to include in the rolling backtest.
# Set to None to automatically backtest ALL available months in the dataset.
BACKTEST_MONTHS = 4   # e.g. 4, 6, 12 … or None for ALL

DATA_ASSETS = {
    "cb-detail-2024" : "Chargeback Detail V2.5 2024.xlsx",
    "cb-detail-0101" : "Chargeback Detail V2.5 0101 to 3110.xlsx",
    "cb-rej-2024"    : "Chargeback Rejections Report V1.1 2024.xlsx",
    "cb-rej-0101"    : "Chargeback Rejections Report V1.1 0101 to 3110.xlsx",
    "sales-2024"     : "Daily Sales Details V1.7 2024.xlsx",
    "sales-0101"     : "Daily Sales Details V1.7 0101 3110.xlsx",
    "cb-detail-1101-1231" : "Chargeback_Detail_V2_5_20251101_20251231.xlsx",
    "cb-rej-1101"    : "Chargeback_Rejections_Report_V1.1_20251101_20251130.xlsx",
    "sales-1101"     : "Daily_Sales_Details_V1.7_PC_20251101_20251130.xlsx",
    "contracts"      : "clean_Contract Pricing Audit V1.3.csv",   # contract start/end/price features
}

CONTRACT_FILE = "clean_Contract Pricing Audit V1.3.csv"   # pre-cleaned combined contract CSV

CB_FILES    = ["Chargeback Detail V2.5 2024.xlsx",
               "Chargeback Detail V2.5 0101 to 3110.xlsx",
               "Chargeback_Detail_V2_5_20251101_20251231.xlsx"
               ]
REJ_FILES   = ["Chargeback Rejections Report V1.1 2024.xlsx",
               "Chargeback Rejections Report V1.1 0101 to 3110.xlsx",
               "Chargeback_Rejections_Report_V1.1_20251101_20251130.xlsx"
               ]
SALES_FILES = ["Daily Sales Details V1.7 2024.xlsx",
               "Daily Sales Details V1.7 0101 3110.xlsx",
               "Daily_Sales_Details_V1.7_PC_20251101_20251130.xlsx"
               ]

CACHE_CB      = "cache_cb_clean.csv"
CACHE_MONTHLY = "cache_monthly_features.csv"
CACHE_SPLITS  = ["X_train.csv","X_val.csv","X_test.csv",
                  "y_train.csv","y_val.csv","y_test.csv"]

# ── Auto-detect data end month from CB_FILES filenames (no hardcoding) ───────
import re as _re
_end_months = []
for _f in CB_FILES:
    _m = _re.search(r"_(\d{4})(\d{2})\d{2}\.xlsx$", _f)
    if _m:
        _end_months.append(f"{_m.group(1)}-{_m.group(2)}")
DATA_END_MONTH = max(_end_months) if _end_months else None
print(f"Data end month     : {DATA_END_MONTH}  (auto-detected from CB_FILES)")

if FORCE_REBUILD:
    import os
    for f in [CACHE_CB, CACHE_MONTHLY] + CACHE_SPLITS:
        if os.path.exists(f):
            os.remove(f)
    print("Cache cleared — will rebuild from raw files")
else:
    print("Cache mode — will load from CSVs if available")

print(f"Backtest window    : {BACKTEST_MONTHS if BACKTEST_MONTHS is not None else 'ALL'} months")
#DATA_END_MONTH = "2025-10"  # override auto-detect

In [ ]:
import os, re, shutil, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# ── Step A: Download files from Azure ML Data Assets ─────────────────────────
missing = [n for n in (CB_FILES + REJ_FILES + SALES_FILES) if not os.path.exists(n)]

if missing:
    print(f"Downloading {len(DATA_ASSETS)} data assets from Azure ML...")
    from azure.ai.ml import MLClient
    from azure.identity import DefaultAzureCredential
    from azureml.core import Workspace, Datastore

    ml_client = MLClient(
        DefaultAzureCredential(),
        subscription_id     = "00a4ad6e-2f6b-49aa-ad1a-acb90a583c37",
        resource_group_name = "Piramal-rg",
        workspace_name      = "Piramal-ML"
    )
    ws = Workspace(
        subscription_id = "00a4ad6e-2f6b-49aa-ad1a-acb90a583c37",
        resource_group  = "Piramal-rg",
        workspace_name  = "Piramal-ML"
    )

    for asset_name, local_name in DATA_ASSETS.items():
        if os.path.exists(local_name):
            print(f"   {local_name} already exists")
            continue

        print(f"  Downloading {asset_name}...")
        asset = ml_client.data.get(asset_name, label="latest")
        path  = asset.path
        print(f"    path: {path}")

        # Fixed regex — handles full azureml://subscriptions/.../datastores/.../paths/... format
        match = re.match(r".*datastores/([^/]+)/paths/(.+)", path)
        if not match:
            print(f"      Cannot parse path: {path}")
            continue

        ds_name, file_prefix = match.group(1), match.group(2)
        print(f"    datastore={ds_name}  prefix={file_prefix}")

        datastore = Datastore.get(ws, ds_name)
        tmp = f"_tmp_{asset_name}"
        os.makedirs(tmp, exist_ok=True)
        datastore.download(target_path=tmp, prefix=file_prefix, overwrite=True)

        found = False
        for root, _, files in os.walk(tmp):
            for fname in files:
                shutil.copy(os.path.join(root, fname), local_name)
                print(f"     saved as {local_name}")
                found = True
                break
            if found:
                break

        if not found:
            print(f"      Nothing found in {tmp}")
        shutil.rmtree(tmp, ignore_errors=True)

    print("\nAll downloads complete!\n")
else:
    print(" All Excel files already present locally\n")

# ── Step B: Load raw files into cb and sales dataframes ──────────────────────
if not os.path.exists(CACHE_CB):
    print("Processing raw files (takes ~2-3 minutes)...")

    def norm_str(s):
        return s.astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
    def norm_ndc(s):
        return s.astype(str).str.strip().str.replace(r"[\s\-]", "", regex=True)

    # Approved chargebacks
    print("  • Approved chargebacks...")
    cb_approved = pd.concat([pd.read_excel(f) for f in CB_FILES], ignore_index=True)
    cb_approved["Chargeback Status"] = "A"
    cb_approved["Agreement_norm"]    = norm_str(cb_approved["Contract Number"].fillna("UNKNOWN"))
    cb_approved["SKU_norm"]          = norm_ndc(cb_approved["NDC Number"].fillna("UNKNOWN"))
    print(f"    {len(cb_approved):,} rows")

    # Rejected chargebacks
    print("  • Rejected chargebacks...")
    cb_rejected = pd.concat([pd.read_excel(f) for f in REJ_FILES], ignore_index=True)
    cb_rejected["Chargeback Status"] = "R"
    cb_rejected["Agreement_norm"]    = norm_str(cb_rejected["Contract Number"].fillna("UNKNOWN"))
    cb_rejected["SKU_norm"]          = norm_ndc(cb_rejected["Item NDC Number"].fillna("UNKNOWN"))
    cb_rejected = cb_rejected.rename(columns={
        "Submitted WAC Price":   "WAC Price",
        "Actual Per Unit Price": "Unit CB Amount",
    })
    print(f"    {len(cb_rejected):,} rows")

    # Combine into one table
    shared_cols = ["Agreement_norm","SKU_norm","Process Date","Chargeback Status",
                   "Chargeback Amount","Chargeback Quantity",
                   "WAC Price","Unit CB Amount",
                   "Member HIN","Member Type","Contract Type","Wholesaler Number",
                   "Wholesaler Name"]
    for col in shared_cols:
        if col not in cb_rejected.columns: cb_rejected[col] = np.nan
        if col not in cb_approved.columns: cb_approved[col] = np.nan

    # Dedup on ALL columns FIRST (real duplicates only), then reduce columns
    cb_approved = cb_approved.drop_duplicates()
    cb_rejected = cb_rejected.drop_duplicates()
    cb_all = pd.concat([cb_approved[shared_cols], cb_rejected[shared_cols]])
    cb_all["Process Date"]        = pd.to_datetime(cb_all["Process Date"],        errors="coerce")
    cb_all["Chargeback Amount"]   = pd.to_numeric(cb_all["Chargeback Amount"],    errors="coerce").clip(lower=0).fillna(0)
    cb_all["Chargeback Quantity"] = pd.to_numeric(cb_all["Chargeback Quantity"],  errors="coerce").fillna(0)
    cb_all["WAC Price"]           = pd.to_numeric(cb_all["WAC Price"],            errors="coerce")
    cb_all["Unit CB Amount"]      = pd.to_numeric(cb_all["Unit CB Amount"],       errors="coerce")
    cb_all["Contract Type"]       = pd.to_numeric(cb_all["Contract Type"],        errors="coerce")
    cb_all["Member Type"]         = pd.to_numeric(cb_all["Member Type"],          errors="coerce")
    cb_all["year_month"]          = cb_all["Process Date"].dt.to_period("M")
    cb_all = cb_all.dropna(subset=["Process Date"]).reset_index(drop=True)
    # Cap to DATA_END_MONTH so stale files with future data never leak in
    if DATA_END_MONTH:
        import pandas as _pd
        cb_all = cb_all[cb_all["year_month"] <= _pd.Period(DATA_END_MONTH, "M")].reset_index(drop=True)
        print(f"  Date cap: kept data up to {DATA_END_MONTH} ({len(cb_all):,} rows remain)")
    print(f"  Combined CB: {len(cb_all):,} rows")

    # Sales
    print("  • Sales data...")
    sales_all = pd.concat([pd.read_excel(f, header=3) for f in SALES_FILES], ignore_index=True)
    sales_all = sales_all.drop_duplicates()
    sales_all["SKU_norm"]      = norm_ndc(sales_all["Item / NDC"].fillna("UNKNOWN"))
    sales_all["Invoice Date"]  = pd.to_datetime(sales_all["Invoice Date"], errors="coerce")
    sales_all["Total Revenue"] = pd.to_numeric(sales_all["Total Revenue"], errors="coerce").fillna(0)
    sales_all["year_month"]    = sales_all["Invoice Date"].dt.to_period("M")
    sales_all = sales_all.dropna(subset=["Invoice Date"])
    print(f"    {len(sales_all):,} rows")

    cb_all.to_csv(CACHE_CB, index=False)
    print(f"\n Saved to cache: {CACHE_CB}")
    cb    = cb_all
    sales = sales_all

else:
    print(f"⚡ Loading CB from cache: {CACHE_CB}")
    cb = pd.read_csv(CACHE_CB)
    cb["Process Date"] = pd.to_datetime(cb["Process Date"])
    cb["year_month"]   = cb["Process Date"].dt.to_period("M")

    def norm_ndc(s):
        return s.astype(str).str.strip().str.replace(r"[\s\-]", "", regex=True)
    sales_all = pd.concat([pd.read_excel(f, header=3) for f in SALES_FILES], ignore_index=True)
    sales_all["SKU_norm"]      = norm_ndc(sales_all["Item / NDC"].fillna("UNKNOWN"))
    sales_all["Invoice Date"]  = pd.to_datetime(sales_all["Invoice Date"], errors="coerce")
    sales_all["Total Revenue"] = pd.to_numeric(sales_all["Total Revenue"], errors="coerce").fillna(0)
    sales_all["year_month"]    = sales_all["Invoice Date"].dt.to_period("M")
    sales = sales_all.dropna(subset=["Invoice Date"])

print(f"\nCB shape   : {cb.shape}")
print(f"Sales shape: {sales.shape}")
print(f"Date range : {cb['Process Date'].min().date()} → {cb['Process Date'].max().date()}")


In [ ]:
# STEP 1.5 — DATA CLEANING
# Runs after raw data is loaded (Cell 3) and before feature engineering (Cell 4).
# Note: negatives, NaT dates, and numeric coercion are already handled in Cell 3.
# This cell covers: deduplication, date range filter, zero-amount rows,
# NDC format validation, WAC price sanity, UNKNOWN pairs, sales cleaning.
# Note: Outlier capping is intentionally skipped — high CB amounts are real
# business activity. Outliers are handled via log1p target transform + XGBoost
# regularization instead.

print("=" * 60)
print("DATA CLEANING REPORT")
print("=" * 60)

cb_start    = len(cb)
sales_start = len(sales)

# 1. DEDUPLICATION
dedup_cols = ["Agreement_norm", "SKU_norm", "Process Date",
              "Chargeback Amount", "Chargeback Status", "Member HIN", "Wholesaler Number"]
cb_dedup_cols = [c for c in dedup_cols if c in cb.columns]
before = len(cb)
#cb = cb.drop_duplicates(subset=cb_dedup_cols).reset_index(drop=True)
print(f"[1] Duplicates removed          : {before - len(cb):,} rows")

# 2. DATE RANGE FILTER
date_min = pd.Timestamp("2020-01-01")
date_max = pd.Timestamp.today()
before = len(cb)
cb = cb[(cb["Process Date"] >= date_min) & (cb["Process Date"] <= date_max)].reset_index(drop=True)
print(f"[2] Out-of-range dates removed  : {before - len(cb):,} rows  (kept: {date_min.date()} to {date_max.date()})")

# 3. ZERO-AMOUNT APPROVED CHARGEBACKS
before = len(cb)
zero_mask = (cb["Chargeback Status"] == "A") & (cb["Chargeback Amount"] == 0)
cb = cb[~zero_mask].reset_index(drop=True)
print(f"[3] Zero-amount approved CB rows: {before - len(cb):,} rows removed")

# 4. NDC FORMAT VALIDATION (informational only — not removing rows)
import re as _re
def is_valid_ndc(ndc_str):
    return bool(_re.match(r"^\d{9,11}$", str(ndc_str)))
invalid_ndc_mask = ~cb["SKU_norm"].apply(is_valid_ndc) & (cb["SKU_norm"] != "UNKNOWN")
n_invalid_ndc = invalid_ndc_mask.sum()
print(f"[4] Invalid NDC format rows     : {n_invalid_ndc:,}  (flagged only — not removed)")
if n_invalid_ndc > 0:
    print(f"    Sample: {cb.loc[invalid_ndc_mask, 'SKU_norm'].value_counts().head(5).to_dict()}")

# 5. WAC PRICE SANITY CHECK
wac_col = "WAC Price"
if wac_col in cb.columns:
    bad_wac = ((cb[wac_col] < 0.01) | (cb[wac_col] > 100_000)) & cb[wac_col].notna()
    n_bad_wac = bad_wac.sum()
    cb.loc[bad_wac, wac_col] = np.nan
    print(f"[5] WAC Price out-of-range to NaN: {n_bad_wac:,} rows  (range: $0.01 to $100,000)")

# 6. BOTH CONTRACT AND NDC ARE UNKNOWN
before = len(cb)
both_unknown = (cb["Agreement_norm"] == "UNKNOWN") & (cb["SKU_norm"] == "UNKNOWN")
cb = cb[~both_unknown].reset_index(drop=True)
print(f"[6] Both contract+NDC UNKNOWN   : {before - len(cb):,} rows removed")

# 7. SALES DATA CLEANING
before_sales = len(sales)
sales = sales[sales["Total Revenue"] > 0].reset_index(drop=True)
print(f"[7] Sales zero/neg revenue rows : {before_sales - len(sales):,} rows removed")

sales_dedup_cols = [c for c in ["SKU_norm","Invoice Date","Total Revenue"] if c in sales.columns]
before_sales = len(sales)
#sales = sales.drop_duplicates(subset=sales_dedup_cols).reset_index(drop=True)
print(f"[7] Sales duplicate rows removed: {before_sales - len(sales):,} rows")

# 8. MISSING VALUE SUMMARY (informational)
key_cols = ["Agreement_norm","SKU_norm","Chargeback Amount","WAC Price","Unit CB Amount","Member HIN"]
key_cols_present = [c for c in key_cols if c in cb.columns]
missing_summary = cb[key_cols_present].isnull().sum()
print(f"\n[8] Missing values in CB after cleaning:")
found_missing = False
for col, n in missing_summary[missing_summary > 0].items():
    print(f"    {col:30s}: {n:,} ({100*n/len(cb):.1f}%)")
    found_missing = True
if not found_missing:
    print("    None — all key columns fully populated!")

# FINAL SUMMARY
print(f"\n{'='*60}")
print(f"CLEANING SUMMARY")
print(f"{'='*60}")
print(f"CB rows   : {cb_start:>10,} -> {len(cb):>10,}  (removed {cb_start - len(cb):,})")
print(f"Sales rows: {sales_start:>10,} -> {len(sales):>10,}  (removed {sales_start - len(sales):,})")
print(f"CB date range (clean): {cb['Process Date'].min().date()} to {cb['Process Date'].max().date()}")
print(f"Unique agreements     : {cb['Agreement_norm'].nunique():,}")
print(f"Unique SKUs           : {cb['SKU_norm'].nunique():,}")
print("Data cleaning complete — proceeding to feature engineering.")

In [ ]:
# STEP 2 — MONTHLY FEATURE ENGINEERING
# Each row = one Agreement-SKU in one specific month
# Target = CB amount for THAT month (not cumulative total)
if not os.path.exists(CACHE_MONTHLY):
    print("Building monthly features...")
    cb_approved = cb[cb["Chargeback Status"] == "A"].copy()
    monthly = cb_approved.groupby(["Agreement_norm", "SKU_norm", "year_month"]).agg(
        cb_amt_sum          = ("Chargeback Amount",  "sum"),
        cb_count            = ("Chargeback Amount",  "count"),
        cb_qty_sum          = ("Chargeback Quantity","sum"),
        avg_wac_price       = ("WAC Price",          "mean"),
        avg_unit_cb_amt     = ("Unit CB Amount",     "mean"),
        unique_member_count = ("Member HIN",         "nunique"),
        wholesaler_count    = ("Wholesaler Number",  "nunique"),
    ).reset_index()

    total_counts = cb.groupby(["Agreement_norm", "SKU_norm", "year_month"]).size().reset_index(name="total_count")
    rej_counts   = cb[cb["Chargeback Status"]=="R"].groupby(
        ["Agreement_norm","SKU_norm","year_month"]).size().reset_index(name="rej_count")
    rej_rate = total_counts.merge(rej_counts, on=["Agreement_norm","SKU_norm","year_month"], how="left")
    rej_rate["rej_count"]      = rej_rate["rej_count"].fillna(0)
    rej_rate["rejection_rate"] = rej_rate["rej_count"] / rej_rate["total_count"].clip(lower=1)
    monthly = monthly.merge(rej_rate[["Agreement_norm","SKU_norm","year_month","rejection_rate"]],
        on=["Agreement_norm","SKU_norm","year_month"], how="left")

    ct = cb_approved.groupby(["Agreement_norm","year_month"])["Contract Type"].agg(
        lambda x: x.mode()[0] if len(x.mode())>0 else np.nan
    ).reset_index().rename(columns={"Contract Type":"contract_type"})
    monthly = monthly.merge(ct, on=["Agreement_norm","year_month"], how="left")

    mt = cb_approved.groupby(["Agreement_norm","year_month"])["Member Type"].agg(
        lambda x: x.mode()[0] if len(x.mode())>0 else np.nan
    ).reset_index().rename(columns={"Member Type":"member_type"})
    monthly = monthly.merge(mt, on=["Agreement_norm","year_month"], how="left")

    sku_cnt = cb_approved.groupby(["Agreement_norm","year_month"])["SKU_norm"].nunique(
    ).reset_index().rename(columns={"SKU_norm":"sku_count_per_agreement"})
    monthly = monthly.merge(sku_cnt, on=["Agreement_norm","year_month"], how="left")

    monthly_sales = sales.groupby(["SKU_norm","year_month"]).agg(
        sales_sum = ("Total Revenue","sum")
    ).reset_index()
    monthly_sales["sales_log"] = np.log1p(monthly_sales["sales_sum"])
    monthly = monthly.merge(monthly_sales, on=["SKU_norm","year_month"], how="left")
    monthly[["sales_sum","sales_log"]] = monthly[["sales_sum","sales_log"]].fillna(0)

    monthly["month_num"] = monthly["year_month"].apply(
        lambda x: x.month if hasattr(x, "month") else pd.Period(str(x), "M").month)

    # SORT before lag features (critical)
    monthly = monthly.sort_values(["Agreement_norm","SKU_norm","year_month"]).reset_index(drop=True)

    # Lag features on CB amount
    grp = monthly.groupby(["Agreement_norm","SKU_norm"])["cb_amt_sum"]
    monthly["cb_lag1"]     = grp.shift(1)
    monthly["cb_lag2"]     = grp.shift(2)
    monthly["cb_lag3"]     = grp.shift(3)
    monthly["cb_rolling3"] = grp.shift(1).transform(lambda x: x.rolling(3, min_periods=1).mean())
    monthly["cb_rolling6"] = grp.shift(1).transform(lambda x: x.rolling(6, min_periods=1).mean())

    # Trend features
    monthly["cb_growth_ratio"]  = monthly["cb_lag1"] / (monthly["cb_rolling6"] + 1)
    monthly["cb_momentum"]      = monthly["cb_lag1"] - monthly["cb_lag3"]

    # Exponential weighted mean: weights recent months more than older ones
    monthly["cb_ewm3"] = monthly.groupby(["Agreement_norm","SKU_norm"])["cb_amt_sum"].transform(lambda x: x.shift(1).ewm(span=3, min_periods=1).mean())

    # How far the exponential trend is above the flat 6-month average
    monthly["cb_ewm_vs_rolling6"] = monthly["cb_ewm3"] / (monthly["cb_rolling6"] + 1)

    # Lag features on activity metrics (previous month only — no leakage)
    monthly["cb_count_lag1"]            = monthly.groupby(["Agreement_norm","SKU_norm"])["cb_count"].shift(1)
    monthly["cb_qty_lag1"]              = monthly.groupby(["Agreement_norm","SKU_norm"])["cb_qty_sum"].shift(1)
    monthly["avg_unit_cb_amt_lag1"]     = monthly.groupby(["Agreement_norm","SKU_norm"])["avg_unit_cb_amt"].shift(1)
    monthly["rejection_rate_lag1"]      = monthly.groupby(["Agreement_norm","SKU_norm"])["rejection_rate"].shift(1)
    monthly["unique_member_count_lag1"] = monthly.groupby(["Agreement_norm","SKU_norm"])["unique_member_count"].shift(1)
    monthly["wholesaler_count_lag1"]    = monthly.groupby(["Agreement_norm","SKU_norm"])["wholesaler_count"].shift(1)

    monthly["cb_log"] = np.log1p(monthly["cb_amt_sum"])
    monthly = monthly.dropna(subset=["cb_lag1"]).reset_index(drop=True)

    fill_cols = [
        "cb_lag1","cb_lag2","cb_lag3","cb_rolling3","cb_rolling6",
        "cb_growth_ratio","cb_momentum",
        "cb_ewm3","cb_ewm_vs_rolling6",
        "cb_count_lag1","cb_qty_lag1","avg_unit_cb_amt_lag1",
        "rejection_rate_lag1","unique_member_count_lag1","wholesaler_count_lag1",
        "sales_sum","sales_log","rejection_rate","contract_type",
        "member_type","avg_wac_price","sku_count_per_agreement"
    ]
    monthly[fill_cols] = monthly[fill_cols].fillna(0)

    # ── Contract features (start/end dates, price) ────────────────────────────
    def build_contract_features(monthly_df, contracts_df):
        """Add 5 contract-based features per Agreement-SKU per month."""
        w = contracts_df[contracts_df["Award Status"] == "W"].copy()
        w["Agreement_norm"] = w["Agreement_norm"].astype(str).str.strip()
        w["SKU_norm"]       = (w["SKU_norm"].astype(str)
                                 .str.replace(".0", "", regex=False)
                                 .str.strip())
        # Parse date columns (they may come in as strings)
        for col in ["Contract Start Date", "Contract End Date",
                    "Contract Extension Date", "effective_end"]:
            if col in w.columns:
                w[col] = pd.to_datetime(w[col], errors="coerce")

        # Base: earliest start, latest effective_end per Agreement+SKU
        base = (w.groupby(["Agreement_norm", "SKU_norm"])
                  .agg(contract_start=("Contract Start Date", "min"),
                       effective_end  =("effective_end",       "max"))
                  .reset_index())

        # has_extension: 1 if there is any row where Extension > End
        ext_flag = (
            w[w["Contract Extension Date"].notna() &
              (w["Contract Extension Date"] > w["Contract End Date"])]
            [["Agreement_norm", "SKU_norm"]]
            .drop_duplicates()
            .assign(has_extension=1)
        )
        base = base.merge(ext_flag, on=["Agreement_norm", "SKU_norm"], how="left")
        base["has_extension"] = base["has_extension"].fillna(0).astype(int)

        monthly_df = monthly_df.merge(base, on=["Agreement_norm", "SKU_norm"], how="left")

        # Time-varying features
        monthly_df["_month_dt"] = monthly_df["year_month"].apply(
            lambda x: pd.Period(str(x), "M").to_timestamp())
        monthly_df["_cur_m"]   = (monthly_df["_month_dt"].dt.year * 12 +
                                   monthly_df["_month_dt"].dt.month)
        monthly_df["_start_m"] = (monthly_df["contract_start"].dt.year * 12 +
                                   monthly_df["contract_start"].dt.month)
        monthly_df["_end_m"]   = (monthly_df["effective_end"].dt.year * 12 +
                                   monthly_df["effective_end"].dt.month)

        monthly_df["months_to_expiry"]    = (monthly_df["_end_m"] - monthly_df["_cur_m"]).clip(lower=0)
        monthly_df["contract_age_months"] = (monthly_df["_cur_m"] - monthly_df["_start_m"]).clip(lower=0)
        monthly_df["is_expiring_soon"]    = (monthly_df["months_to_expiry"] <= 3).astype(int)
        monthly_df.drop(columns=["_month_dt","_cur_m","_start_m","_end_m",
                                  "contract_start","effective_end"], inplace=True)

        # Contract price: expand price rows across valid months, then merge
        w_prices = w[["Agreement_norm", "SKU_norm",
                       "Contract Price", "Item Start Date", "Item End Date"]].copy()
        w_prices = w_prices.dropna(subset=["Contract Price", "Item Start Date"])
        w_prices["Item Start Date"] = pd.to_datetime(w_prices["Item Start Date"], errors="coerce")
        w_prices["Item End Date"]   = pd.to_datetime(w_prices["Item End Date"],   errors="coerce")
        # Cap 2080 placeholder dates to 2030
        w_prices["Item End Date"]   = w_prices["Item End Date"].clip(upper=pd.Timestamp("2030-12-31"))
        w_prices = w_prices.dropna(subset=["Item Start Date", "Item End Date"])

        # Expand each price row to one row per month it covers
        rows = []
        for _, r in w_prices.iterrows():
            months = pd.period_range(
                pd.Period(r["Item Start Date"], "M"),
                pd.Period(r["Item End Date"],   "M"), freq="M")
            for m in months:
                rows.append({
                    "Agreement_norm": r["Agreement_norm"],
                    "SKU_norm":       r["SKU_norm"],
                    "year_month":     str(m),
                    "contract_price": r["Contract Price"],
                })
        if rows:
            price_df = pd.DataFrame(rows).drop_duplicates(
                subset=["Agreement_norm", "SKU_norm", "year_month"], keep="last")
            # Ensure year_month is string on both sides (monthly may be Period[M])
            monthly_df["year_month"] = monthly_df["year_month"].astype(str)
            price_df["year_month"]   = price_df["year_month"].astype(str)
            monthly_df = monthly_df.merge(
                price_df, on=["Agreement_norm", "SKU_norm", "year_month"], how="left")
        else:
            monthly_df["contract_price"] = 0.0

        # Fill all 5 new columns with 0 for unmatched rows
        for col in ["months_to_expiry", "contract_age_months",
                    "is_expiring_soon", "has_extension", "contract_price"]:
            if col not in monthly_df.columns:
                monthly_df[col] = 0
            monthly_df[col] = monthly_df[col].fillna(0)

        return monthly_df

    # Load contract CSV and apply features
    if os.path.exists(CONTRACT_FILE):
        print("[2b] Loading contract data and building contract features...")
        contracts_df = pd.read_csv(CONTRACT_FILE, low_memory=False)
        monthly = build_contract_features(monthly, contracts_df)
        print(f"     Contract features added: months_to_expiry, contract_age_months, "
              f"is_expiring_soon, has_extension, contract_price")
        print(f"     Rows with contract_price > 0 : {(monthly['contract_price'] > 0).sum():,}")
    else:
        print(f"[2b] WARNING: {CONTRACT_FILE} not found — contract features will be 0")
        for col in ["months_to_expiry", "contract_age_months",
                    "is_expiring_soon", "has_extension", "contract_price"]:
            monthly[col] = 0

    monthly.to_csv(CACHE_MONTHLY, index=False)
    print(f"Monthly features saved to {CACHE_MONTHLY}")

else:
    print(f"Loading monthly features from cache: {CACHE_MONTHLY}")
    monthly = pd.read_csv(CACHE_MONTHLY)
    monthly["year_month"] = monthly["year_month"].astype(str)

print(f"\nMonthly dataset : {monthly.shape[0]:,} rows x {monthly.shape[1]} cols")
print(f"Date range      : {monthly['year_month'].min()} -> {monthly['year_month'].max()}")
print(f"Unique pairs    : {monthly.groupby(['Agreement_norm','SKU_norm']).ngroups:,}")
print(f"Unique months   : {monthly['year_month'].nunique()}")
print(f"\nTarget (cb_amt_sum) stats:")
print(monthly["cb_amt_sum"].describe().round(2))

In [ ]:
# STEP 3 — DEFINE FEATURES + TIME-BASED TRAIN/VAL/TEST SPLIT
# All activity features now use PREVIOUS month values — no leakage

feature_cols = [
    "cb_lag1",
    "cb_lag2",
    "cb_lag3",
    "cb_rolling3",
    "cb_rolling6",
    "cb_growth_ratio",
    "cb_momentum",
    "cb_ewm3",           
    "cb_ewm_vs_rolling6",
    "cb_count_lag1",
    "cb_qty_lag1",
    "avg_unit_cb_amt_lag1",
    "rejection_rate_lag1",
    "unique_member_count_lag1",
    "wholesaler_count_lag1",
    "contract_type",
    "member_type",
    "avg_wac_price",
    "sku_count_per_agreement",
    "sales_sum",
    "sales_log",
    "month_num",
    # Contract features (new — require clean_Contract Pricing Audit V1.3.csv)
    "months_to_expiry",       # months until contract expires (0 if unknown)
    "contract_age_months",    # how many months the contract has been running
    "is_expiring_soon",       # 1 if expiring within 3 months, else 0
    "has_extension",          # 1 if contract was previously extended, else 0
    "contract_price",         # negotiated unit price under this agreement
]

print(f"Total features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

X = monthly[feature_cols].fillna(0)
y = monthly["cb_log"].values

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y range: {y.min():.2f} -> {y.max():.2f}  (log scale)")

if not os.path.exists("X_train.csv"):
    all_months = sorted(monthly["year_month"].unique())
    n = len(all_months)

    train_end = all_months[int(n * 0.70)]
    val_end   = all_months[int(n * 0.85)]

    train_mask = monthly["year_month"] <  train_end
    val_mask   = (monthly["year_month"] >= train_end) & (monthly["year_month"] <  val_end)
    test_mask  = monthly["year_month"] >= val_end

    X_train = X[train_mask];  y_train = y[train_mask]
    X_val   = X[val_mask];    y_val   = y[val_mask]
    X_test  = X[test_mask];   y_test  = y[test_mask]

    print(f"\nTrain: {X_train.shape}  |  months: {all_months[0]} -> {train_end}")
    print(f"Val  : {X_val.shape}    |  months: {train_end} -> {val_end}")
    print(f"Test : {X_test.shape}   |  months: {val_end} -> {all_months[-1]}")

    X_train.to_csv("X_train.csv", index=False)
    X_val.to_csv("X_val.csv",     index=False)
    X_test.to_csv("X_test.csv",   index=False)
    import pandas as pd
    pd.Series(y_train, name="cb_log").to_csv("y_train.csv", index=False)
    pd.Series(y_val,   name="cb_log").to_csv("y_val.csv",   index=False)
    pd.Series(y_test,  name="cb_log").to_csv("y_test.csv",  index=False)
    print("Train/val/test splits saved to CSV")

else:
    X_train = pd.read_csv("X_train.csv")
    X_val   = pd.read_csv("X_val.csv")
    X_test  = pd.read_csv("X_test.csv")
    y_train = pd.read_csv("y_train.csv")["cb_log"].values
    y_val   = pd.read_csv("y_val.csv")["cb_log"].values
    y_test  = pd.read_csv("y_test.csv")["cb_log"].values
    print("Loaded splits from CSV")
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
# ==============================================================================
# V7 — STACKED GENERALIZATION WITH PAIR-SPECIFIC ADAPTATION
# ==============================================================================
# The most aggressive ML architecture for pair-level accuracy.
#
# 5 Layers:
#   1. Base Model Zoo (21 models — diverse architectures)
#   2. Pair-Specific Meta-Features (historical error profiles per pair)
#   3. Stacked Generalization (XGBoost learns to combine all models)
#   4. Online Bias Correction (fix systematic per-pair errors)
#   5. Confidence-Weighted Blending (safety net for volatile pairs)
#
# Run after Cell 5 (needs: monthly, feature_cols, top_5_params, cb)
# ==============================================================================
import pandas as pd, numpy as np, warnings
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error
warnings.filterwarnings("ignore")

# Best params from HyperDrive
top_5_params = [
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.0196,
     "subsample": 0.681, "colsample_bytree": 0.932, "min_child_weight": 1,
     "gamma": 0.126, "reg_alpha": 0.939, "reg_lambda": 0.153},
]

print("=" * 80)
print("V7 — STACKED GENERALIZATION PIPELINE")
print("=" * 80)

# ==============================================================================
# STEP 1: BUILD EXTENDED FEATURES
# ==============================================================================
print("\n[1/5] Building extended features...")

def build_extended_features(df):
    out = df.copy()
    
    # Ratio features
    out["lag1_over_lag2"] = out["cb_lag1"] / out["cb_lag2"].clip(lower=1)
    out["lag1_over_rolling6"] = out["cb_lag1"] / out["cb_rolling6"].clip(lower=1)
    out["ewm_over_rolling6"] = out["cb_ewm3"] / out["cb_rolling6"].clip(lower=1)
    
    # Volatility
    grp = out.groupby(["Agreement_norm", "SKU_norm"])["cb_amt_sum"]
    out["cb_std3"] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=2).std()).fillna(0)
    out["cb_std6"] = grp.transform(lambda x: x.shift(1).rolling(6, min_periods=3).std()).fillna(0)
    out["cv_recent"] = out["cb_std3"] / out["cb_ewm3"].clip(lower=1)
    
    # Cross-sectional
    sku_total = out.groupby(["SKU_norm", "year_month"])["cb_amt_sum"].sum().reset_index()
    sku_total = sku_total.rename(columns={"cb_amt_sum": "sku_total_cb"}).sort_values(["SKU_norm", "year_month"])
    sku_total["sku_total_lag1"] = sku_total.groupby("SKU_norm")["sku_total_cb"].shift(1)
    out = out.merge(sku_total[["SKU_norm", "year_month", "sku_total_lag1"]],
                    on=["SKU_norm", "year_month"], how="left")
    
    agr_total = out.groupby(["Agreement_norm", "year_month"])["cb_amt_sum"].sum().reset_index()
    agr_total = agr_total.rename(columns={"cb_amt_sum": "agr_total_cb"}).sort_values(["Agreement_norm", "year_month"])
    agr_total["agr_total_lag1"] = agr_total.groupby("Agreement_norm")["agr_total_cb"].shift(1)
    out = out.merge(agr_total[["Agreement_norm", "year_month", "agr_total_lag1"]],
                    on=["Agreement_norm", "year_month"], how="left")
    
    out["pair_share_sku"] = out["cb_ewm3"] / out["sku_total_lag1"].clip(lower=1)
    out["pair_share_agr"] = out["cb_ewm3"] / out["agr_total_lag1"].clip(lower=1)
    
    # Count change
    out["count_change"] = out["cb_count_lag1"] - out.groupby(
        ["Agreement_norm", "SKU_norm"])["cb_count_lag1"].shift(1)
    
    # Pair history length
    out["pair_history"] = out.groupby(["Agreement_norm", "SKU_norm"]).cumcount()
    
    # Recent min/max ratio (how spiky is recent data)
    out["recent_min"] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).min()).fillna(0)
    out["recent_max"] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).max()).fillna(0)
    out["recent_range_ratio"] = (out["recent_max"] - out["recent_min"]) / out["cb_ewm3"].clip(lower=1)
    
    fill_cols = ["lag1_over_lag2", "lag1_over_rolling6", "ewm_over_rolling6",
                 "cb_std3", "cb_std6", "cv_recent", "sku_total_lag1", "agr_total_lag1",
                 "pair_share_sku", "pair_share_agr", "count_change", "pair_history",
                 "recent_min", "recent_max", "recent_range_ratio"]
    out[fill_cols] = out[fill_cols].fillna(0)
    for col in ["lag1_over_lag2", "lag1_over_rolling6", "ewm_over_rolling6", "recent_range_ratio"]:
        out[col] = out[col].clip(-10, 10)
    
    return out

monthly_ext = build_extended_features(monthly)

extended_features = feature_cols + [
    "lag1_over_lag2", "lag1_over_rolling6", "ewm_over_rolling6",
    "cb_std3", "cb_std6", "cv_recent", "sku_total_lag1", "agr_total_lag1",
    "pair_share_sku", "pair_share_agr", "count_change", "pair_history",
    "recent_range_ratio",
]

print(f"  Extended features: {len(extended_features)}")

# ==============================================================================
# STEP 2: DEFINE BASE MODEL ZOO
# ==============================================================================
print("[2/5] Defining base model zoo...")

def get_base_models():
    return {
        "XGB_HD": XGBRegressor(**top_5_params[0], random_state=42, tree_method="hist", n_jobs=-1),
        "XGB_shallow": XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            gamma=0.2, reg_alpha=0.5, reg_lambda=2.0, random_state=42, tree_method="hist", n_jobs=-1),
        "XGB_deep": XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.01,
            subsample=0.7, colsample_bytree=0.6, min_child_weight=1,
            gamma=0.1, reg_alpha=0.1, reg_lambda=0.5, random_state=42, tree_method="hist", n_jobs=-1),
        "XGB_reg": XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.6, colsample_bytree=0.5, min_child_weight=10,
            gamma=0.5, reg_alpha=2.0, reg_lambda=5.0, random_state=42, tree_method="hist", n_jobs=-1),
        "RF_deep": RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5,
            max_features=0.7, random_state=42, n_jobs=-1),
        "RF_shallow": RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=10,
            max_features=0.5, random_state=42, n_jobs=-1),
        "GBR": GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, min_samples_leaf=10, random_state=42),
        "Ridge": Ridge(alpha=1.0),
    }

# ==============================================================================
# STEP 3: WALK-FORWARD WITH STACKING
# ==============================================================================
print("[3/5] Running walk-forward stacking backtest...")
print()

WALKFORWARD_MIN_TRAIN = 6
all_months = sorted(monthly_ext["year_month"].unique())

if BACKTEST_MONTHS is None:
    candidate_months = all_months[WALKFORWARD_MIN_TRAIN:]
else:
    n_bt = min(BACKTEST_MONTHS, len(all_months) - 1)
    candidate_months = all_months[-n_bt:]

backtest_months = [m for m in candidate_months
                   if all_months.index(m) >= WALKFORWARD_MIN_TRAIN]

# Storage for bias correction (Layer 4)
pair_error_history = {}  # {(agr, sku): [(predicted, actual), ...]}

all_v7_results = []

for test_month in backtest_months:
    idx = all_months.index(test_month)
    input_month = all_months[idx - 1]
    
    train_data = monthly_ext[monthly_ext["year_month"] < test_month].copy()
    input_rows = monthly_ext[monthly_ext["year_month"] == input_month].copy()
    target_rows = monthly_ext[monthly_ext["year_month"] == test_month].copy()
    
    if len(train_data) == 0 or len(input_rows) == 0 or len(target_rows) == 0:
        continue
    
    # ==================================================================
    # LAYER 1: Train all base models and generate OOF predictions
    # ==================================================================
    # We need out-of-fold predictions on training data for the stacker
    # Use the last 2 months of training data as the stacking validation set
    train_months = sorted(train_data["year_month"].unique())
    if len(train_months) < 4:
        continue
    
    stack_val_months = train_months[-2:]
    stack_train = train_data[~train_data["year_month"].isin(stack_val_months)]
    stack_val = train_data[train_data["year_month"].isin(stack_val_months)]
    
    X_stack_train = stack_train[extended_features].fillna(0)
    y_stack_train = stack_train["cb_log"].values
    X_stack_val = stack_val[extended_features].fillna(0)
    X_input = input_rows[extended_features].fillna(0)
    
    base_models = get_base_models()
    
    # Train each base model, get OOF predictions and input predictions
    oof_preds = {}  # predictions on stack_val (for training stacker)
    input_preds = {}  # predictions on input_rows (for final prediction)
    
    for name, model in base_models.items():
        model.fit(X_stack_train, y_stack_train)
        oof_preds[name] = np.expm1(model.predict(X_stack_val)).clip(min=0)
        input_preds[name] = np.expm1(model.predict(X_input)).clip(min=0)
    
    # Add deterministic baselines
    for df, preds_dict, label in [
        (stack_val, oof_preds, "oof"),
        (input_rows, input_preds, "input")
    ]:
        preds_dict["last_month"] = df["cb_lag1"].values.clip(min=0)
        preds_dict["ewm"] = df["cb_ewm3"].values.clip(min=0)
        preds_dict["rolling3"] = df["cb_rolling3"].values.clip(min=0)
        preds_dict["rolling6"] = df["cb_rolling6"].values.clip(min=0)
    
    # Also add ratio-based ML prediction
    # Train ratio model
    ratio_train = stack_train[stack_train["cb_lag1"] > 0].copy()
    if len(ratio_train) > 50:
        ratio_train["ratio_target"] = (
            ratio_train["cb_amt_sum"] / ratio_train["cb_lag1"].clip(lower=1)
        ).clip(0, 5)
        ratio_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.8, min_child_weight=3,
            random_state=42, tree_method="hist", n_jobs=-1)
        ratio_model.fit(ratio_train[extended_features].fillna(0), ratio_train["ratio_target"])
        
        oof_ratio = ratio_model.predict(X_stack_val).clip(0.1, 5.0)
        oof_preds["ratio_ml"] = (stack_val["cb_lag1"].values * oof_ratio).clip(min=0)
        
        input_ratio = ratio_model.predict(X_input).clip(0.1, 5.0)
        input_preds["ratio_ml"] = (input_rows["cb_lag1"].values * input_ratio).clip(min=0)
    
    # Residual-based ML prediction
    stack_train_res = stack_train.copy()
    stack_train_res["residual"] = stack_train_res["cb_amt_sum"] - stack_train_res["cb_ewm3"]
    res_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.2, reg_alpha=1.0, reg_lambda=2.0,
        random_state=42, tree_method="hist", n_jobs=-1)
    res_model.fit(stack_train[extended_features].fillna(0), stack_train_res["residual"])
    
    oof_preds["residual_ml"] = (stack_val["cb_ewm3"].values + res_model.predict(X_stack_val)).clip(min=0)
    input_preds["residual_ml"] = (input_rows["cb_ewm3"].values + res_model.predict(X_input)).clip(min=0)
    
    all_base_names = list(oof_preds.keys())
    
    # ==================================================================
    # LAYER 2: Build pair-specific meta-features
    # ==================================================================
    def build_meta_features(df, preds_dict, pair_error_hist):
        """Build stacker input: base model preds + pair-specific features."""
        meta = pd.DataFrame()
        
        # All base model predictions as features
        for name in all_base_names:
            meta[f"pred_{name}"] = preds_dict[name]
        
        # Spread between models (disagreement = uncertainty)
        pred_cols = [f"pred_{n}" for n in all_base_names]
        meta["model_spread"] = meta[pred_cols].std(axis=1)
        meta["model_spread_pct"] = meta["model_spread"] / meta[pred_cols].mean(axis=1).clip(lower=1)
        meta["model_max"] = meta[pred_cols].max(axis=1)
        meta["model_min"] = meta[pred_cols].min(axis=1)
        meta["model_range_ratio"] = (meta["model_max"] - meta["model_min"]) / meta[pred_cols].median(axis=1).clip(lower=1)
        
        # Pair-specific features from the dataframe
        meta["cv_recent"] = df["cv_recent"].values
        meta["cb_std3"] = df["cb_std3"].values
        meta["pair_history"] = df["pair_history"].values
        meta["recent_range_ratio"] = df["recent_range_ratio"].values
        meta["lag1_over_rolling6"] = df["lag1_over_rolling6"].values
        meta["ewm_over_rolling6"] = df["ewm_over_rolling6"].values
        meta["pair_share_sku"] = df["pair_share_sku"].values
        meta["pair_share_agr"] = df["pair_share_agr"].values
        meta["cb_momentum"] = df["cb_momentum"].values
        meta["cb_growth_ratio"] = df["cb_growth_ratio"].values
        meta["month_num"] = df["month_num"].values
        
        # Historical error profile per pair (from previous backtest months)
        meta["hist_bias"] = 0.0
        meta["hist_abs_err"] = 0.0
        meta["hist_n_months"] = 0
        
        for i, (_, row) in enumerate(df[["Agreement_norm", "SKU_norm"]].iterrows()):
            key = (str(row["Agreement_norm"]), str(row["SKU_norm"]))
            if key in pair_error_hist and len(pair_error_hist[key]) > 0:
                errors = pair_error_hist[key]
                recent = errors[-3:]  # last 3 months
                biases = [a - p for p, a in recent]
                meta.loc[i, "hist_bias"] = np.mean(biases)
                meta.loc[i, "hist_abs_err"] = np.mean([abs(b) for b in biases])
                meta.loc[i, "hist_n_months"] = len(recent)
        
        return meta.fillna(0)
    
    # Build stacker training data
    oof_meta = build_meta_features(stack_val, oof_preds, pair_error_history)
    oof_actual = stack_val["cb_amt_sum"].values
    
    # Build stacker input data
    input_meta = build_meta_features(input_rows, input_preds, pair_error_history)
    
    # ==================================================================
    # LAYER 3: Train the stacker
    # ==================================================================
    stacker = XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.5, reg_lambda=1.0,
        random_state=42, tree_method="hist", n_jobs=-1
    )
    
    # Train stacker to predict ACTUAL amount (not log)
    stacker.fit(oof_meta, oof_actual)
    
    # Stacker prediction
    stacker_pred = stacker.predict(input_meta).clip(min=0)
    
    # Also train a second stacker in log space for comparison
    stacker_log = XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.5, reg_lambda=1.0,
        random_state=42, tree_method="hist", n_jobs=-1
    )
    stacker_log.fit(oof_meta, np.log1p(oof_actual))
    stacker_log_pred = np.expm1(stacker_log.predict(input_meta)).clip(min=0)
    
    # Blend both stackers (raw + log space)
    stacker_blend = 0.5 * stacker_pred + 0.5 * stacker_log_pred
    
    # ==================================================================
    # LAYER 4: Online Bias Correction
    # ==================================================================
    corrected_pred = stacker_blend.copy()
    
    for i, (_, row) in enumerate(input_rows[["Agreement_norm", "SKU_norm"]].iterrows()):
        key = (str(row["Agreement_norm"]), str(row["SKU_norm"]))
        if key in pair_error_history and len(pair_error_history[key]) >= 2:
            recent = pair_error_history[key][-3:]
            biases = [a - p for p, a in recent]
            avg_bias = np.mean(biases)
            # Only correct if bias is consistent (same direction)
            if all(b > 0 for b in biases) or all(b < 0 for b in biases):
                corrected_pred[i] = max(0, corrected_pred[i] + avg_bias * 0.5)
            elif abs(avg_bias) > 0.2 * corrected_pred[i]:
                corrected_pred[i] = max(0, corrected_pred[i] + avg_bias * 0.3)
    
    # ==================================================================
    # LAYER 5: Confidence-Weighted Blending
    # ==================================================================
    final_pred = corrected_pred.copy()
    
    pred_matrix = np.column_stack([input_preds[n] for n in all_base_names])
    model_spread = pred_matrix.std(axis=1)
    model_median = np.median(pred_matrix, axis=1)
    spread_ratio = model_spread / np.clip(model_median, 1, None)
    
    # For high-uncertainty pairs (models disagree >50%), blend with EWM
    high_uncertainty = spread_ratio > 0.8
    if high_uncertainty.any():
        ewm_vals = input_rows["cb_ewm3"].values.clip(min=0)
        final_pred[high_uncertainty] = (
            0.8 * corrected_pred[high_uncertainty] +
            0.2 * ewm_vals[high_uncertainty]
        )
    
    # Cap extreme predictions: no prediction should be >3x the pair's historical max
    for i, (_, row) in enumerate(input_rows[["Agreement_norm", "SKU_norm"]].iterrows()):
        pair_data = train_data[
            (train_data["Agreement_norm"] == row["Agreement_norm"]) &
            (train_data["SKU_norm"] == row["SKU_norm"])
        ]["cb_amt_sum"]
        if len(pair_data) > 0:
            hist_max = pair_data.max()
            if final_pred[i] > 5 * hist_max:
                final_pred[i] = min(final_pred[i], 3 * hist_max)
    
    # ==================================================================
    # COLLECT RESULTS
    # ==================================================================
    merged = input_rows[["Agreement_norm", "SKU_norm"]].copy().reset_index(drop=True)
    merged["month"] = test_month
    merged["pred_v7_stacker"] = stacker_blend
    merged["pred_v7_corrected"] = corrected_pred
    merged["pred_v7_final"] = final_pred
    
    # Also store base models for comparison
    merged["pred_global"] = input_preds["XGB_HD"]
    merged["pred_last_month"] = input_preds["last_month"]
    merged["pred_ewm"] = input_preds["ewm"]
    merged["pred_meta"] = input_preds.get("ratio_ml", input_preds["ewm"])  # best from TEST-3
    
    # Actuals
    actuals = target_rows[["Agreement_norm", "SKU_norm", "cb_amt_sum"]].rename(
        columns={"cb_amt_sum": "actual"})
    merged = merged.merge(actuals, on=["Agreement_norm", "SKU_norm"], how="inner")
    
    all_v7_results.append(merged)
    
    # Update error history for Layer 4 (next month's bias correction)
    for _, row in merged.iterrows():
        key = (str(row["Agreement_norm"]), str(row["SKU_norm"]))
        if key not in pair_error_history:
            pair_error_history[key] = []
        pair_error_history[key].append((row["pred_v7_final"], row["actual"]))
    
    # Print progress
    r2_v7 = r2_score(merged["actual"], merged["pred_v7_final"])
    r2_glob = r2_score(merged["actual"], merged["pred_global"])
    r2_ewm = r2_score(merged["actual"], merged["pred_ewm"])
    print(f"  {test_month} | V7={r2_v7:.4f} | Global={r2_glob:.4f} | EWM={r2_ewm:.4f}")

    # Print stacker feature importance (first month only)
    if test_month == backtest_months[0]:
        fi = pd.Series(stacker.feature_importances_, index=oof_meta.columns)
        fi = fi.sort_values(ascending=False).head(15)
        print(f"\n  Stacker top features:")
        for fname, fval in fi.items():
            print(f"    {fname:30s}: {fval:.4f}")
        print()

# ==============================================================================
# COMPREHENSIVE RESULTS
# ==============================================================================
v7 = pd.concat(all_v7_results, ignore_index=True)
v7.to_csv("v7_results.csv", index=False)

print(f"\n{'='*80}")
print("V7 RESULTS — ALL APPROACHES COMPARED")
print(f"{'='*80}")

models_to_compare = ["pred_global", "pred_last_month", "pred_ewm",
                      "pred_v7_stacker", "pred_v7_corrected", "pred_v7_final"]

print(f"\n  {'Model':25s} {'R2':>8} {'MAE ($)':>12} {'Port.Err':>10} "
      f"{'<10%':>6} {'<15%':>6} {'<20%':>6} {'<30%':>6} {'>30%':>6}")
print(f"  {'-'*95}")

for col in models_to_compare:
    name = col.replace("pred_", "")
    r2 = r2_score(v7["actual"], v7[col])
    mae = mean_absolute_error(v7["actual"], v7[col])
    port_err = 100 * (v7[col].sum() - v7["actual"].sum()) / v7["actual"].sum()
    
    pair_err = v7.groupby(["Agreement_norm", "SKU_norm"]).apply(
        lambda g: 100 * abs(g["actual"].sum() - g[col].sum()) / max(g["actual"].sum(), 1)
    )
    pct_10 = (pair_err <= 10).mean() * 100
    pct_15 = (pair_err <= 15).mean() * 100
    pct_20 = (pair_err <= 20).mean() * 100
    pct_30 = (pair_err <= 30).mean() * 100
    pct_bad = (pair_err > 30).mean() * 100
    
    print(f"  {name:25s} {r2:>8.4f} ${mae:>10,.0f} {port_err:>+9.1f}% "
          f"{pct_10:>5.0f}% {pct_15:>5.0f}% {pct_20:>5.0f}% {pct_30:>5.0f}% {pct_bad:>5.0f}%")

# ==============================================================================
# BY VALUE BUCKET
# ==============================================================================
train_all = monthly_ext[monthly_ext["year_month"] < backtest_months[0]].copy()
pair_val = (
    train_all.groupby(["Agreement_norm", "SKU_norm"])["cb_amt_sum"]
    .mean().reset_index().rename(columns={"cb_amt_sum": "avg_cb"})
    .sort_values("avg_cb", ascending=False).reset_index(drop=True)
)
pair_val["rank_pct"] = np.arange(1, len(pair_val)+1) / len(pair_val)
pair_val["bucket"] = np.where(
    pair_val["rank_pct"] <= 0.10, "top_10",
    np.where(pair_val["rank_pct"] <= 0.20, "top_10_20",
    np.where(pair_val["rank_pct"] <= 0.50, "top_20_50", "bottom_50")))

v7 = v7.merge(pair_val[["Agreement_norm", "SKU_norm", "bucket"]],
              on=["Agreement_norm", "SKU_norm"], how="left")
v7["bucket"] = v7["bucket"].fillna("bottom_50")

print(f"\n{'='*80}")
print("V7 — BY VALUE BUCKET")
print(f"{'='*80}")

for bkt in ["top_10", "top_10_20", "top_20_50", "bottom_50"]:
    bd = v7[v7["bucket"] == bkt]
    if len(bd) < 3:
        continue
    n_pairs = bd.groupby(["Agreement_norm", "SKU_norm"]).ngroups
    total_cb = bd["actual"].sum()
    
    print(f"\n  --- {bkt.upper()} ({n_pairs} pairs, ${total_cb:,.0f}) ---")
    print(f"  {'Model':25s} {'R2':>8} {'<10%':>6} {'<15%':>6} {'<30%':>6} {'>30%':>6} {'Port.Err':>10}")
    print(f"  {'-'*75}")
    
    for col in models_to_compare:
        name = col.replace("pred_", "")
        r2 = r2_score(bd["actual"], bd[col]) if len(bd) > 2 else 0
        port_err = 100 * (bd[col].sum() - bd["actual"].sum()) / max(bd["actual"].sum(), 1)
        pair_err = bd.groupby(["Agreement_norm", "SKU_norm"]).apply(
            lambda g: 100 * abs(g["actual"].sum() - g[col].sum()) / max(g["actual"].sum(), 1)
        )
        pct_10 = (pair_err <= 10).mean() * 100
        pct_15 = (pair_err <= 15).mean() * 100
        pct_30 = (pair_err <= 30).mean() * 100
        pct_bad = (pair_err > 30).mean() * 100
        print(f"  {name:25s} {r2:>8.4f} {pct_10:>5.0f}% {pct_15:>5.0f}% "
              f"{pct_30:>5.0f}% {pct_bad:>5.0f}% {port_err:>+9.1f}%")

# ==============================================================================
# TOP 30 PAIRS DETAIL
# ==============================================================================
print(f"\n{'='*80}")
print("V7 — TOP 30 PAIRS DETAIL")
print(f"{'='*80}")

pair_detail = v7.groupby(["Agreement_norm", "SKU_norm", "bucket"]).agg(
    actual=("actual", "sum"),
    v7_final=("pred_v7_final", "sum"),
    global_pred=("pred_global", "sum"),
    ewm_pred=("pred_ewm", "sum"),
    months=("month", "count"),
).reset_index().sort_values("actual", ascending=False).reset_index(drop=True)

pair_detail["v7_err"] = 100 * abs(pair_detail["v7_final"] - pair_detail["actual"]) / pair_detail["actual"].clip(lower=1)
pair_detail["global_err"] = 100 * abs(pair_detail["global_pred"] - pair_detail["actual"]) / pair_detail["actual"].clip(lower=1)

print(f"\n  {'#':>3} {'Agreement':>15} {'SKU':>15} {'Actual':>12} "
      f"{'V7':>12} {'V7 Err':>8} {'Global':>12} {'Glob Err':>8} {'Better?':>8}")
print(f"  {'-'*105}")

v7_wins = 0
for i, (_, row) in enumerate(pair_detail.head(30).iterrows()):
    better = "V7" if row["v7_err"] < row["global_err"] else "GLOBAL"
    if better == "V7":
        v7_wins += 1
    print(f"  {i+1:>3} {str(row['Agreement_norm']):>15} {str(row['SKU_norm']):>15} "
          f"${row['actual']:>10,.0f} ${row['v7_final']:>10,.0f} {row['v7_err']:>6.1f}% "
          f"${row['global_pred']:>10,.0f} {row['global_err']:>6.1f}% {better:>8}")

print(f"\n  V7 wins: {v7_wins}/30 top pairs")

print(f"\nSaved to v7_results.csv")


In [ ]:
# ==============================================================================
# V7 — FORWARD PREDICTION
# ==============================================================================
import pandas as pd, numpy as np, warnings
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
warnings.filterwarnings("ignore")

latest_month = monthly_ext["year_month"].max()
pred_month = str(pd.Period(latest_month, "M") + 1)
print(f"Latest data month : {latest_month}")
print(f"Predicting month  : {pred_month}")

# Use all data for training, last 2 months as stacker validation
all_train = monthly_ext.copy()
train_months = sorted(all_train["year_month"].unique())
stack_val_months = train_months[-2:]
stack_train = all_train[~all_train["year_month"].isin(stack_val_months)]
stack_val = all_train[all_train["year_month"].isin(stack_val_months)]
input_rows = monthly_ext[monthly_ext["year_month"] == latest_month].copy()

X_stack_train = stack_train[extended_features].fillna(0)
y_stack_train = stack_train["cb_log"].values
X_stack_val = stack_val[extended_features].fillna(0)
X_input = input_rows[extended_features].fillna(0)

print(f"Stack train: {len(stack_train):,} rows")
print(f"Stack val:   {len(stack_val):,} rows")
print(f"Input rows:  {len(input_rows):,} rows")

# LAYER 1: Train all base models
base_models = get_base_models()
oof_preds = {}
input_preds = {}

for name, model in base_models.items():
    model.fit(X_stack_train, y_stack_train)
    oof_preds[name] = np.expm1(model.predict(X_stack_val)).clip(min=0)
    input_preds[name] = np.expm1(model.predict(X_input)).clip(min=0)

# Deterministic baselines
for df, preds_dict in [(stack_val, oof_preds), (input_rows, input_preds)]:
    preds_dict["last_month"] = df["cb_lag1"].values.clip(min=0)
    preds_dict["ewm"] = df["cb_ewm3"].values.clip(min=0)
    preds_dict["rolling3"] = df["cb_rolling3"].values.clip(min=0)
    preds_dict["rolling6"] = df["cb_rolling6"].values.clip(min=0)

# Ratio ML
ratio_train = stack_train[stack_train["cb_lag1"] > 0].copy()
if len(ratio_train) > 50:
    ratio_train["ratio_target"] = (
        ratio_train["cb_amt_sum"] / ratio_train["cb_lag1"].clip(lower=1)
    ).clip(0, 5)
    ratio_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.8, min_child_weight=3,
        random_state=42, tree_method="hist", n_jobs=-1)
    ratio_model.fit(ratio_train[extended_features].fillna(0), ratio_train["ratio_target"])
    oof_preds["ratio_ml"] = (stack_val["cb_lag1"].values * ratio_model.predict(X_stack_val).clip(0.1, 5.0)).clip(min=0)
    input_preds["ratio_ml"] = (input_rows["cb_lag1"].values * ratio_model.predict(X_input).clip(0.1, 5.0)).clip(min=0)

# Residual ML
stack_train_res = stack_train.copy()
stack_train_res["residual"] = stack_train_res["cb_amt_sum"] - stack_train_res["cb_ewm3"]
res_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.03,
    subsample=0.7, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.2, reg_alpha=1.0, reg_lambda=2.0,
    random_state=42, tree_method="hist", n_jobs=-1)
res_model.fit(stack_train[extended_features].fillna(0), stack_train_res["residual"])
oof_preds["residual_ml"] = (stack_val["cb_ewm3"].values + res_model.predict(X_stack_val)).clip(min=0)
input_preds["residual_ml"] = (input_rows["cb_ewm3"].values + res_model.predict(X_input)).clip(min=0)

all_base_names = list(oof_preds.keys())
print(f"Base models: {len(all_base_names)}")

# LAYER 2 + 3: Build meta-features and train stacker
oof_meta = build_meta_features(stack_val, oof_preds, pair_error_history)
input_meta = build_meta_features(input_rows, input_preds, pair_error_history)

stacker = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.5, reg_lambda=1.0,
    random_state=42, tree_method="hist", n_jobs=-1)
stacker.fit(oof_meta, stack_val["cb_amt_sum"].values)
stacker_pred = stacker.predict(input_meta).clip(min=0)

stacker_log = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.5, reg_lambda=1.0,
    random_state=42, tree_method="hist", n_jobs=-1)
stacker_log.fit(oof_meta, np.log1p(stack_val["cb_amt_sum"].values))
stacker_log_pred = np.expm1(stacker_log.predict(input_meta)).clip(min=0)

stacker_blend = 0.5 * stacker_pred + 0.5 * stacker_log_pred

# LAYER 4: Pair-level bias correction
corrected = stacker_blend.copy()
for i, (_, row) in enumerate(input_rows[["Agreement_norm", "SKU_norm"]].iterrows()):
    key = (str(row["Agreement_norm"]), str(row["SKU_norm"]))
    if key in pair_error_history and len(pair_error_history[key]) >= 2:
        recent = pair_error_history[key][-3:]
        biases = [a - p for p, a in recent]
        avg_bias = np.mean(biases)
        if all(b > 0 for b in biases) or all(b < 0 for b in biases):
            corrected[i] = max(0, corrected[i] + avg_bias * 0.5)
        elif abs(avg_bias) > 0.2 * corrected[i]:
            corrected[i] = max(0, corrected[i] + avg_bias * 0.3)

# LAYER 5: Confidence blending + cap
final = corrected.copy()
pred_matrix = np.column_stack([input_preds[n] for n in all_base_names])
spread_ratio = pred_matrix.std(axis=1) / np.clip(np.median(pred_matrix, axis=1), 1, None)

high_uncertainty = spread_ratio > 0.8
if high_uncertainty.any():
    ewm_vals = input_rows["cb_ewm3"].values.clip(min=0)
    final[high_uncertainty] = 0.8 * corrected[high_uncertainty] + 0.2 * ewm_vals[high_uncertainty]

for i, (_, row) in enumerate(input_rows[["Agreement_norm", "SKU_norm"]].iterrows()):
    pair_data = all_train[
        (all_train["Agreement_norm"] == row["Agreement_norm"]) &
        (all_train["SKU_norm"] == row["SKU_norm"])
    ]["cb_amt_sum"]
    if len(pair_data) > 0:
        hist_max = pair_data.max()
        if final[i] > 5 * hist_max:
            final[i] = min(final[i], 3 * hist_max)

# Confidence intervals
pair_std = all_train.groupby(["Agreement_norm", "SKU_norm"]).apply(
    lambda g: (g["cb_amt_sum"] - g["cb_ewm3"]).std() if len(g) > 2 else g["cb_amt_sum"].std()
).reset_index().rename(columns={0: "hist_std"})

input_with_std = input_rows[["Agreement_norm", "SKU_norm"]].merge(
    pair_std, on=["Agreement_norm", "SKU_norm"], how="left"
)
hist_std = input_with_std["hist_std"].fillna(0).values

# Build output
output = input_rows[["Agreement_norm", "SKU_norm"]].copy().reset_index(drop=True)
output["Agreement_norm"] = output["Agreement_norm"].astype(str).str.replace(".0", "", regex=False)
output["SKU_norm"] = output["SKU_norm"].astype(str).str.replace(".0", "", regex=False)
output["predicted_cb_amt"] = final
output["pred_low"] = (final - 1.5 * hist_std).clip(min=0)
output["pred_high"] = final + 1.5 * hist_std

total_pred = output["predicted_cb_amt"].sum()
print(f"\n{'='*65}")
print(f"V7 FORWARD PREDICTION — {pred_month}")
print(f"{'='*65}")
print(f"Total Predicted CB : ${total_pred:,.2f}")
print(f"Confidence Range   : ${output['pred_low'].sum():,.2f} — ${output['pred_high'].sum():,.2f}")
print(f"Total Pairs        : {len(output)}")

print(f"\nTop 20 Predicted Pairs:")
top20 = output.nlargest(20, "predicted_cb_amt")
print(f"  {'Agreement':>15} {'SKU':>15} {'Predicted':>14} {'Low':>14} {'High':>14}")
print(f"  {'-'*75}")
for _, row in top20.iterrows():
    print(f"  {row['Agreement_norm']:>15} {row['SKU_norm']:>15} "
          f"${row['predicted_cb_amt']:>12,.0f} ${row['pred_low']:>12,.0f} ${row['pred_high']:>12,.0f}")

# ADD NEW PAIRS (only pairs active in 2+ of last 3 months)
last_3 = sorted(monthly["year_month"].unique())[-3:]
recent_activity = (
    monthly[monthly["year_month"].isin(last_3)]
    .groupby(["Agreement_norm", "SKU_norm"])["year_month"]
    .nunique().reset_index().rename(columns={"year_month": "active_months"})
)
eligible = recent_activity[recent_activity["active_months"] >= 2]

predicted_pairs = input_rows[["Agreement_norm", "SKU_norm"]].drop_duplicates()
new_pairs = eligible.merge(predicted_pairs, on=["Agreement_norm", "SKU_norm"],
                           how="left", indicator=True)
new_pairs = new_pairs[new_pairs["_merge"] == "left_only"].drop(columns="_merge")

new_pairs_total = 0
if len(new_pairs) > 0:
    hist_avg = (
        monthly[monthly["year_month"].isin(last_3)]
        .groupby(["Agreement_norm", "SKU_norm"])["cb_amt_sum"]
        .mean().reset_index().rename(columns={"cb_amt_sum": "hist_pred"})
    )
    new_pairs = new_pairs.merge(hist_avg, on=["Agreement_norm", "SKU_norm"], how="left")
    new_pairs["hist_pred"] = new_pairs["hist_pred"].fillna(0)
    new_pairs_total = new_pairs["hist_pred"].sum()
    print(f"\nNew pairs added: {len(new_pairs)} (active 2+ of last 3 months)")
    print(f"New pairs CB: ${new_pairs_total:,.0f}")
else:
    print(f"\nNo new pairs to add")

final_total = total_pred + new_pairs_total
print(f"\n{'='*65}")
print(f"FINAL PREDICTION — {pred_month}")
print(f"{'='*65}")
print(f"  V7 stacker ({len(output)} pairs):  ${total_pred:,.0f}")
print(f"  New pairs (historical):  ${new_pairs_total:,.0f}")
print(f"  TOTAL:                   ${final_total:,.0f}")

# Save
output.to_csv(f"predictions_{pred_month}_v7.csv", index=False)
print(f"\nSaved to predictions_{pred_month}_v7.csv")

In [ ]:
import pandas as pd, numpy as np

v7 = pd.read_csv("v7_results.csv")
v7["Agreement_norm"] = v7["Agreement_norm"].astype(str).str.replace(".0", "", regex=False)
v7["SKU_norm"] = v7["SKU_norm"].astype(str).str.replace(".0", "", regex=False)

pair = v7.groupby(["Agreement_norm", "SKU_norm"]).agg(
    actual=("actual", "sum"),
    v7_pred=("pred_v7_final", "sum"),
    global_pred=("pred_global", "sum"),
    months=("month", "count"),
).reset_index()
pair["v7_err"] = 100 * abs(pair["v7_pred"] - pair["actual"]) / pair["actual"].clip(lower=1)
pair["global_err"] = 100 * abs(pair["global_pred"] - pair["actual"]) / pair["actual"].clip(lower=1)
pair["avg_monthly"] = pair["actual"] / pair["months"]
pair = pair.sort_values("v7_err")

print("=" * 110)
print("BEST 100 PAIRS (lowest V7 error)")
print("=" * 110)
print(f"  {'#':>3} {'Agreement':>15} {'SKU':>15} {'Avg $/mo':>10} {'Actual':>12} "
      f"{'V7 Pred':>12} {'V7 Err':>8} {'Glob Err':>9} {'Months':>6}")
print(f"  {'-'*100}")
for i, (_, r) in enumerate(pair.head(100).iterrows()):
    print(f"  {i+1:>3} {r['Agreement_norm']:>15} {r['SKU_norm']:>15} "
          f"${r['avg_monthly']:>8,.0f} ${r['actual']:>10,.0f} ${r['v7_pred']:>10,.0f} "
          f"{r['v7_err']:>6.1f}% {r['global_err']:>7.1f}% {r['months']:>5}")

b100 = pair.head(200)
print(f"\n  Best 100 summary:")
print(f"    Avg V7 error:     {b100['v7_err'].mean():.1f}%")
print(f"    Avg Global error: {b100['global_err'].mean():.1f}%")
print(f"    Total CB:         ${b100['actual'].sum():,.0f} ({100*b100['actual'].sum()/pair['actual'].sum():.1f}%)")

print(f"\n{'=' * 110}")
print("WORST 100 PAIRS (highest V7 error)")
print("=" * 110)
print(f"  {'#':>3} {'Agreement':>15} {'SKU':>15} {'Avg $/mo':>10} {'Actual':>12} "
      f"{'V7 Pred':>12} {'V7 Err':>8} {'Glob Err':>9} {'Months':>6}")
print(f"  {'-'*100}")
worst = pair.tail(100).iloc[::-1]  # reverse so worst is first
for i, (_, r) in enumerate(worst.iterrows()):
    print(f"  {i+1:>3} {r['Agreement_norm']:>15} {r['SKU_norm']:>15} "
          f"${r['avg_monthly']:>8,.0f} ${r['actual']:>10,.0f} ${r['v7_pred']:>10,.0f} "
          f"{r['v7_err']:>6.1f}% {r['global_err']:>7.1f}% {r['months']:>5}")

w100 = pair.tail(200)
print(f"\n  Worst 100 summary:")
print(f"    Avg V7 error:     {w100['v7_err'].mean():.1f}%")
print(f"    Avg Global error: {w100['global_err'].mean():.1f}%")
print(f"    Total CB:         ${w100['actual'].sum():,.0f} ({100*w100['actual'].sum()/pair['actual'].sum():.1f}%)")
print(f"    Avg monthly CB:   ${w100['avg_monthly'].mean():,.0f}")

print(f"\n{'=' * 110}")
print("DISTRIBUTION SUMMARY")
print("=" * 110)
for threshold in [5, 10, 15, 20, 30, 50]:
    n = (pair["v7_err"] <= threshold).sum()
    dollars = pair[pair["v7_err"] <= threshold]["actual"].sum()
    print(f"  Within {threshold:>2}%: {n:>4} pairs ({100*n/len(pair):.0f}%) | "
          f"${dollars:>14,.0f} ({100*dollars/pair['actual'].sum():.1f}% of CB)")

In [ ]:
import pandas as pd

v7 = pd.read_csv("v7_results.csv")
pair = v7.groupby(["Agreement_norm", "SKU_norm"]).agg(
    total_actual=("actual", "sum"),
    total_v7=("pred_v7_final", "sum"),
    total_global=("pred_global", "sum"),
    months=("month", "count"),
).reset_index()
pair["avg_monthly"] = pair["total_actual"] / pair["months"]
pair["v7_err"] = 100 * abs(pair["total_v7"] - pair["total_actual"]) / pair["total_actual"].clip(lower=1)
pair["global_err"] = 100 * abs(pair["total_global"] - pair["total_actual"]) / pair["total_actual"].clip(lower=1)

tiers = [
    (">$100K/month", 100000),
    ("$50K-$100K", 50000),
    ("$25K-$50K", 25000),
    ("$10K-$25K", 10000),
    ("$5K-$10K", 5000),
    ("$1K-$5K", 1000),
    ("<$1K", 0),
]

print(f"V7 ACCURACY BY BUSINESS TIER")
print(f"{'='*100}")
print(f"  {'Tier':>15} {'Pairs':>6} {'CB $':>14} {'%CB':>6} "
      f"{'<10%':>6} {'<15%':>6} {'<20%':>6} {'>30%':>6} {'Avg V7':>8} {'Avg Glob':>9}")
print(f"  {'-'*90}")

for i, (label, lower) in enumerate(tiers):
    upper = tiers[i-1][1] if i > 0 else float("inf")
    if i == 0:
        tier = pair[pair["avg_monthly"] >= lower]
    else:
        tier = pair[(pair["avg_monthly"] >= lower) & (pair["avg_monthly"] < upper)]
    
    if len(tier) == 0:
        continue
    
    n = len(tier)
    dollars = tier["total_actual"].sum()
    pct_cb = 100 * dollars / pair["total_actual"].sum()
    pct_10 = 100 * (tier["v7_err"] <= 10).mean()
    pct_15 = 100 * (tier["v7_err"] <= 15).mean()
    pct_20 = 100 * (tier["v7_err"] <= 20).mean()
    pct_bad = 100 * (tier["v7_err"] > 30).mean()
    avg_v7 = tier["v7_err"].mean()
    avg_glob = tier["global_err"].mean()
    
    print(f"  {label:>15} {n:>5} ${dollars:>12,.0f} {pct_cb:>5.1f}% "
          f"{pct_10:>5.0f}% {pct_15:>5.0f}% {pct_20:>5.0f}% {pct_bad:>5.0f}% "
          f"{avg_v7:>6.1f}% {avg_glob:>7.1f}%")

# Also show the >$100K pairs individually
print(f"\n{'='*100}")
print(f"TOP 15 PAIRS (>$100K/month) — INDIVIDUAL")
print(f"{'='*100}")
top15 = pair[pair["avg_monthly"] >= 100000].sort_values("total_actual", ascending=False)
print(f"  {'Agreement':>15} {'SKU':>15} {'Avg $/mo':>10} {'Actual':>12} "
      f"{'V7':>12} {'V7 Err':>8} {'Glob Err':>9} {'Status':>10}")
print(f"  {'-'*100}")

for _, r in top15.iterrows():
    status = "GOOD" if r["v7_err"] <= 10 else ("OK" if r["v7_err"] <= 20 else "BAD")
    print(f"  {str(r['Agreement_norm']):>15} {str(r['SKU_norm']):>15} "
          f"${r['avg_monthly']:>8,.0f} ${r['total_actual']:>10,.0f} ${r['total_v7']:>10,.0f} "
          f"{r['v7_err']:>6.1f}% {r['global_err']:>7.1f}% {status:>10}")

good = (top15["v7_err"] <= 10).sum()
ok = ((top15["v7_err"] > 10) & (top15["v7_err"] <= 20)).sum()
bad = (top15["v7_err"] > 20).sum()
print(f"\n  GOOD (<10%): {good}")
print(f"  OK (10-20%): {ok}")
print(f"  BAD (>20%):  {bad}")

In [ ]:
# Load your raw CB data — adjust the column names to match yours
# You're looking for two date columns: when the sale happened and when the CB was filed

# Something like:
cb_raw = pd.read_excel("Chargeback Detail V2.5 2024.xlsx")

# First, just look at what columns you have
print(cb_raw.columns.tolist())

In [ ]:
# Convert date columns to datetime
cb_raw['Process Date'] = pd.to_datetime(cb_raw['Process Date'])
cb_raw['Wholesaler Invoice Date'] = pd.to_datetime(cb_raw['Wholesaler Invoice Date'])

# Calculate the lag in days
cb_raw['cb_lag_days'] = (cb_raw['Process Date'] - cb_raw['Wholesaler Invoice Date']).dt.days

# Drop negatives or nulls (bad data)
lag = cb_raw['cb_lag_days'].dropna()
lag = lag[lag >= 0]

# See the distribution
print("=== CB Lag Distribution (days) ===")
print(lag.describe())
print()
print(f"Within 30 days: {(lag <= 30).mean():.1%}")
print(f"Within 60 days: {(lag <= 60).mean():.1%}")
print(f"Within 90 days: {(lag <= 90).mean():.1%}")
print()

# Show in buckets
bins = [0, 15, 30, 45, 60, 90, 120, 999]
labels = ['0-15', '16-30', '31-45', '46-60', '61-90', '91-120', '120+']
cb_raw['lag_bucket'] = pd.cut(lag, bins=bins, labels=labels)
print(cb_raw['lag_bucket'].value_counts().sort_index())

In [ ]:
# ============================================================
# EXCEL MODEL REPLICATION — Compare against V7
# ============================================================
import pandas as pd, numpy as np

# --- NDC -> Product Group mapping (from Master sheet) ---
ndc_to_prod = {
    '66794015701': 'GABLOFEN', '66794015702': 'GABLOFEN', '66794015101': 'GABLOFEN',
    '66794015502': 'GABLOFEN', '66794015602': 'GABLOFEN', '66794015501': 'GABLOFEN',
    '66794015601': 'GABLOFEN',
    '66794025841': 'Pantoprazole',
    '66794023741': 'DOXYCYCLINE',
    '66794020541': 'GLYCOPYRROLATE', '66794020342': 'GLYCOPYRROLATE',
    '66794020242': 'GLYCOPYRROLATE', '66794020442': 'GLYCOPYRROLATE',
    '66794024942': 'CHLORPROMAZINE', '66794025042': 'CHLORPROMAZINE',
    '66794001525': 'SEVOFLURANE', '66794002225': 'NOVA - SEVOFLURANE',
    '66794001725': 'ISOFLURANE', '66794001710': 'ISOFLURANE',
    '66794001925': 'NOVA - ISOFLURANE', '66794001910': 'NOVA - ISOFLURANE',
    '66794025542': 'Zinc Sulfate', '66794023942': 'Zinc Sulfate', 
    '66794024042': 'Zinc Sulfate',
    '66794021943': 'LINEZOLID', '66794023643': 'NOVA - LINEZOLID',
    '66794023042': 'DEXMED', '66794023541': 'DEXMED',
    '66794023342': 'NOVA - DEXMED', '66794023444': 'DEXMED',
    '66794022841': 'ROCURONIUM', '66794022941': 'ROCURONIUM',
    '66794016002': 'MITIGO', '66794016202': 'MITIGO',
    '66794025964': 'EDARAVONE',
    '66794023242': 'SUCCINYCHOLINE',
}

# --- 1. Load Gross Sales (both files) ---
gross1 = pd.read_excel("Gross Sales V1.9 0101 3110.xlsx")
gross2 = pd.read_excel("Gross_Sales_V1.9_20251101_20251231.xlsx")
gross = pd.concat([gross1, gross2], ignore_index=True)

gross["Shipped Date"] = pd.to_datetime(gross["Shipped Date"], errors="coerce")
gross["year_month"] = gross["Shipped Date"].dt.to_period("M").astype(str)
gross["NDC_clean"] = gross["NDC Number"].astype(str).str.strip()
gross["Prod_Grp"] = gross["NDC_clean"].map(ndc_to_prod)

# Filter to CB products only (ones that have a mapping)
gross_cb = gross[gross["Prod_Grp"].notna()].copy()
print(f"Gross sales: {len(gross)} rows, CB products: {len(gross_cb)} rows")
print(f"Date range: {gross_cb['year_month'].min()} to {gross_cb['year_month'].max()}")

monthly_sales = gross_cb.groupby(["Prod_Grp", "year_month"]).agg(
    shipped_qty=("Shipped Quantity", "sum")
).reset_index()

# --- 2. Load CB actuals (cb should be in memory from Cell 3) ---
cb_prod = cb[cb["Chargeback Status"] == "A"].copy()
cb_prod["NDC_clean"] = cb_prod["SKU_norm"].astype(str).str.replace(".0", "", regex=False).str.strip()
cb_prod["Prod_Grp"] = cb_prod["NDC_clean"].map(ndc_to_prod)
cb_prod = cb_prod[cb_prod["Prod_Grp"].notna()].copy()
cb_prod["year_month"] = cb_prod["Process Date"].dt.to_period("M").astype(str)

monthly_cb = cb_prod.groupby(["Prod_Grp", "year_month"]).agg(
    cb_qty=("Chargeback Quantity", "sum"),
    cb_amt=("Chargeback Amount", "sum")
).reset_index()

print(f"CB data: {monthly_cb['year_month'].min()} to {monthly_cb['year_month'].max()}")

# --- 3. Merge and apply Excel model formula ---
merged = monthly_sales.merge(monthly_cb, on=["Prod_Grp", "year_month"], how="inner")
merged = merged.sort_values(["Prod_Grp", "year_month"]).reset_index(drop=True)
merged["cb_ratio"] = merged["cb_qty"] / merged["shipped_qty"].clip(lower=1)
merged["cb_per_unit"] = merged["cb_amt"] / merged["cb_qty"].clip(lower=1)

# Apply rolling 3-month average formula
excel_results = []
for prod in sorted(merged["Prod_Grp"].unique()):
    prod_data = merged[merged["Prod_Grp"] == prod].sort_values("year_month").reset_index(drop=True)
    for i in range(len(prod_data)):
        if i < 2:  # need at least 2 prior months
            continue
        row = prod_data.iloc[i]
        prior = prod_data.iloc[max(0, i-3):i]
        est_cb_ratio = prior["cb_ratio"].mean()
        est_cb_per_unit = prior["cb_per_unit"].mean()
        excel_pred = row["shipped_qty"] * est_cb_ratio * est_cb_per_unit
        excel_results.append({
            "Prod_Grp": prod, "year_month": row["year_month"],
            "shipped_qty": row["shipped_qty"],
            "excel_pred": excel_pred, "actual_cb": row["cb_amt"],
        })

excel_df = pd.DataFrame(excel_results)
excel_df["excel_err"] = 100 * (excel_df["excel_pred"] - excel_df["actual_cb"]) / excel_df["actual_cb"]

# --- 4. Merge with V7 results ---
v7 = pd.read_csv("v7_results.csv")
v7["SKU_norm"] = v7["SKU_norm"].astype(str).str.replace(".0", "", regex=False).str.strip()
v7["Prod_Grp"] = v7["SKU_norm"].map(ndc_to_prod).fillna("UNKNOWN")

v7_by_prod = v7.groupby(["Prod_Grp", "month"]).agg(
    v7_pred=("pred_v7_final", "sum"),
    ewm_pred=("pred_ewm", "sum"),
    pairs=("actual", "count")
).reset_index().rename(columns={"month": "year_month"})

comparison = excel_df.merge(
    v7_by_prod[["Prod_Grp", "year_month", "v7_pred", "ewm_pred", "pairs"]],
    on=["Prod_Grp", "year_month"], how="left"
)
comparison["v7_err"] = 100 * (comparison["v7_pred"] - comparison["actual_cb"]) / comparison["actual_cb"]

# --- 5. Print results ---
print(f"\n{'='*110}")
print("EXCEL MODEL vs V7 — BY PRODUCT GROUP")
print(f"{'='*110}")

for prod in sorted(comparison["Prod_Grp"].unique()):
    pd_data = comparison[comparison["Prod_Grp"] == prod].sort_values("year_month")
    print(f"\n--- {prod} (Total Actual: ${pd_data['actual_cb'].sum():,.0f}) ---")
    print(f"  {'Month':>8} {'Actual':>12} {'Excel Pred':>12} {'Excel Err':>10} {'V7 Pred':>12} {'V7 Err':>10}")
    print(f"  {'-'*70}")
    for _, r in pd_data.iterrows():
        v7_str = f"${r['v7_pred']:>10,.0f}" if pd.notna(r.get('v7_pred')) else f"{'—':>11}"
        v7_err_str = f"{r['v7_err']:>+8.1f}%" if pd.notna(r.get('v7_err')) else f"{'—':>9}"
        print(f"  {r['year_month']:>8} ${r['actual_cb']:>10,.0f} ${r['excel_pred']:>10,.0f} "
              f"{r['excel_err']:>+8.1f}% {v7_str} {v7_err_str}")

# --- 6. Overall comparison (overlapping months) ---
overlap = comparison[comparison["v7_pred"].notna()].copy()
print(f"\n{'='*110}")
print(f"OVERALL — OVERLAPPING MONTHS ({sorted(overlap['year_month'].unique())})")
print(f"{'='*110}")
print(f"  Total Actual:      ${overlap['actual_cb'].sum():>12,.0f}")
print(f"  Total Excel Pred:  ${overlap['excel_pred'].sum():>12,.0f}  "
      f"({100*(overlap['excel_pred'].sum()-overlap['actual_cb'].sum())/overlap['actual_cb'].sum():+.1f}%)")
print(f"  Total V7 Pred:     ${overlap['v7_pred'].sum():>12,.0f}  "
      f"({100*(overlap['v7_pred'].sum()-overlap['actual_cb'].sum())/overlap['actual_cb'].sum():+.1f}%)")
print(f"\n  Avg Absolute Error:")
print(f"    Excel model: {overlap['excel_err'].abs().mean():.1f}%")
print(f"    V7:          {overlap['v7_err'].abs().mean():.1f}%")

comparison.to_csv("excel_vs_v7_comparison.csv", index=False)
print(f"\nSaved to excel_vs_v7_comparison.csv") 